> Notebook-friendly copy of `appendix/02-ssh.ipynb`, generated by `tools/make_live.py`. Edit the book notebook, not this file.

# SSH: Secure Remote Access

ssh (secure shell) is the protocol underneath two things you will do constantly: authenticating to github without typing a password, and logging into a remote machine — a university server, a computing cluster — to run something too heavy for your laptop. Both uses share the same mechanism: a keypair, generated once.

**🎯 Learning objectives**

- Explain what ssh keys are and why they replace a password.
- Generate an ssh keypair and add the public half to github.
- Connect to a remote server with ssh, and copy files to and from it.
- Manage several hosts and keys with an ssh config file.

## What ssh actually does

ssh opens an encrypted connection between two machines and authenticates you to the remote one — normally with a password, or, better, with a **keypair**. A keypair is two mathematically linked files: a *private* key, which never leaves your machine, and a *public* key, which you hand out freely to anything you want to trust you. The remote side (github, a cluster login node) challenges your machine to prove it holds the private key matching a public key already on file; if the proof succeeds, you are in, with no password ever sent over the network. This is both more secure than a password and more convenient — nothing secret crosses the network, and there is nothing to retype.

## Generating a keypair

```bash
# macOS / Linux
ssh-keygen -t ed25519 -C "you@example.com"
# press Enter to accept the default location (~/.ssh/id_ed25519),
# and set a passphrase if you want an extra layer of protection
```

```powershell
# Windows (PowerShell, built-in OpenSSH)
ssh-keygen -t ed25519 -C "you@example.com"
# key lives in %USERPROFILE%\.ssh\id_ed25519
```

This creates two files: `id_ed25519` (private — keep it secret, never share it, never commit it) and `id_ed25519.pub` (public — safe to share; this is what you hand to github or a server administrator).

**ℹ️ Never share your private key**

The file without `.pub` in its name is private. It should never be emailed, committed to a repository, or copied to a shared drive. If you ever suspect it was exposed, generate a new keypair and remove the old public key from everywhere it was added.

## Authenticating to github

Add the contents of `id_ed25519.pub` to github, under Settings -> SSH and GPG keys -> New SSH key. Then:

```bash
ssh -T git@github.com            # test the connection
git clone git@github.com:org/repo.git
```

The test command should greet you by username rather than asking for a password — that confirms the keypair is working.

## Connecting to a remote server

The same keypair works for logging into any server that has your public key on file — most commonly a university or HPC login node for thesis-scale computation.

```bash
ssh your-username@cluster.example.edu
```

Once connected, you have a terminal on the remote machine: the navigation and file commands from the [terminal-commands page](01-terminal-commands.ipynb) work exactly the same there. To move files between your laptop and the remote machine without a full session, use `scp` for single files, or `rsync` for whole directories (it only transfers what actually changed):

```bash
scp results.csv your-username@cluster.example.edu:~/project/
rsync -av local_folder/ your-username@cluster.example.edu:~/remote_folder/
```

**🧠 Computational-thinking fundamental: the private key proves who you are**

Everything about ssh security rests on one file staying secret: your private key. Anyone who obtains it can authenticate as you, to github or to any server that trusts your public key, with no further proof required. It should exist in exactly one place: your own machine's `~/.ssh/` folder.

<details>
<summary><b>🔍 Going deeper: ssh-agent</b></summary>

An ssh-agent caches your unlocked key for the session, so you do not retype a passphrase on every connection.

```bash
eval "$(ssh-agent -s)"
ssh-add ~/.ssh/id_ed25519
```

</details>

<details>
<summary><b>🔍 Going deeper: managing several hosts with ~/.ssh/config</b></summary>

One block per remote turns a long command into a short name:

```text
Host github.com
    User git
    IdentityFile ~/.ssh/id_ed25519

Host cluster
    HostName cluster.example.edu
    User your-username
    IdentityFile ~/.ssh/id_ed25519
```

With this in place, `ssh cluster` replaces `ssh your-username@cluster.example.edu`, and the right key is picked automatically for each host.

</details>

<details>
<summary><b>🔍 Going deeper: running Jupyter on a remote machine</b></summary>

A cluster's login node often has no browser, but you can still use its Jupyter through a tunnel: start the notebook server remotely on a fixed port, then forward that port to your laptop.

```bash
# on the remote machine
jupyter notebook --no-browser --port=8888

# on your laptop, in a new terminal
ssh -L 8888:localhost:8888 your-username@cluster.example.edu
```

Open `localhost:8888` in your own browser; the notebook is running remotely, but reachable through the tunnel exactly as if it were local.

</details>

**📌 Takeaways**

- ssh authenticates with a keypair instead of a password: a private key that never leaves your machine, and a public key you share freely.
- Generate one with `ssh-keygen -t ed25519`, then add the `.pub` file to github or a server's authorized keys.
- The same mechanism connects to github and to any remote server, including a thesis-relevant computing cluster.
- `scp`/`rsync` move files over the same connection; a `~/.ssh/config` file shortens repeated connections to named hosts.
- Never let your private key leave your own machine.

## Resources

- [GitHub Docs — Connecting to GitHub with SSH](https://docs.github.com/en/authentication/connecting-to-github-with-ssh) — github's own reference for generating a key and adding it to your account.
- [Project Pythia — Getting started with GitHub](https://foundations.projectpythia.org/foundations/getting-started-github/) — geoscience-oriented walkthrough that includes ssh setup alongside github basics.